In [ ]:
# 📦 Install required packages
!pip install rdflib networkx matplotlib gradio pandas langchain huggingface_hub openai tiktoken

In [ ]:
# 📚 Imports
import rdflib
from rdflib import Graph, RDF, Namespace, Literal
import networkx as nx
import matplotlib.pyplot as plt
import gradio as gr
import pandas as pd
import requests
from io import BytesIO
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpoint
from langchain.schema import Document


In [ ]:
# 🔐 API Token (replace with your real token if needed)
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your-hf-token"

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1",
    temperature=0.5,
    generation_kwargs={"max_length": 512}
)


In [ ]:
# 🧬 Build RDF Graph
CT = Namespace("http://clinicaltrials.org/")
g = Graph()
g.bind("ct", CT)

# Add trial data
g.add((CT.Trial001, RDF.type, CT.ClinicalTrial))
g.add((CT.Trial001, CT.studyDisease, CT.BreastCancer))
g.add((CT.Trial001, CT.hasIntervention, CT.DrugA))
g.add((CT.Trial001, CT.primaryOutcome, Literal("Survival Rate")))

g.add((CT.Trial002, RDF.type, CT.ClinicalTrial))
g.add((CT.Trial002, CT.studyDisease, CT.LungCancer))
g.add((CT.Trial002, CT.hasIntervention, CT.DrugB))
g.add((CT.Trial002, CT.primaryOutcome, Literal("Tumor Reduction")))


In [ ]:
# 🔎 Query RDF Graph
def query_trials_by_disease(disease):
    results = []
    for s, p, o in g.triples((None, CT.studyDisease, None)):
        if disease.lower() in str(o).lower():
            trial_id = str(s).split("/")[-1]
            intervention = g.value(s, CT.hasIntervention)
            outcome = g.value(s, CT.primaryOutcome)
            results.append({
                "Trial": trial_id,
                "Intervention": str(intervention).split("/")[-1],
                "Primary Outcome": str(outcome)
            })
    return pd.DataFrame(results)


In [ ]:
# 🌐 Fetch clinical trials from ClinicalTrials.gov
def fetch_trials_ctgov(condition):
    url = f"https://clinicaltrials.gov/api/query/study_fields?expr={condition}&fields=NCTId,Condition,InterventionName,BriefTitle&min_rnk=1&max_rnk=5&fmt=json"
    response = requests.get(url)
    data = response.json()
    trials = data['StudyFieldsResponse']['StudyFields']
    results = []
    for trial in trials:
        results.append({
            "NCT ID": trial.get("NCTId", [""])[0],
            "Condition": trial.get("Condition", [""])[0],
            "Intervention": trial.get("InterventionName", [""])[0],
            "Title": trial.get("BriefTitle", [""])[0]
        })
    return pd.DataFrame(results)


In [ ]:
# 📊 Graph visualization
def rdf_to_networkx(rdf_graph):
    nxg = nx.DiGraph()
    for subj, pred, obj in rdf_graph:
        s = str(subj).split("/")[-1]
        p = str(pred).split("/")[-1]
        o = str(obj).split("/")[-1] if isinstance(obj, rdflib.URIRef) else str(obj)
        nxg.add_edge(s, o, label=p)
    return nxg

def draw_graph():
    G = rdf_to_networkx(g)
    pos = nx.spring_layout(G, seed=42)
    edge_labels = nx.get_edge_attributes(G, 'label')

    fig, ax = plt.subplots(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=2800, font_size=10, ax=ax)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='darkred', ax=ax)

    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    return buf


In [ ]:
# 🧠 Build RAG-style QA chain from RDF graph
qa_chain = None

def build_rag_from_graph():
    global qa_chain
    docs = []
    for s, p, o in g:
        triple = f"{s} {p} {o}"
        docs.append(Document(page_content=triple))

    splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    texts = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(texts, embeddings)

    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())
    return "RAG chain built from graph."


In [ ]:
# 💬 Ask questions about graph
def ask_graph_question(question):
    if qa_chain:
        return qa_chain.run(question)
    return "Please build the RAG chain first."


In [ ]:
# 🎨 Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## 🧪 Clinical Trials Knowledge Graph (Live + RAG)")

    with gr.Tab("Query RDF by Disease"):
        disease_input = gr.Textbox(label="Disease Name")
        query_btn = gr.Button("Query Trials")
        rdf_output = gr.Dataframe(label="Matching RDF Trials")
        query_btn.click(fn=query_trials_by_disease, inputs=disease_input, outputs=rdf_output)

    with gr.Tab("Live Fetch ClinicalTrials.gov"):
        cond_input = gr.Textbox(label="Condition")
        fetch_btn = gr.Button("Fetch Trials")
        live_output = gr.Dataframe(label="Live Trials")
        fetch_btn.click(fn=fetch_trials_ctgov, inputs=cond_input, outputs=live_output)

    with gr.Tab("RAG Graph QA"):
        build_btn = gr.Button("Build RAG")
        build_output = gr.Textbox(label="Status")
        build_btn.click(fn=build_rag_from_graph, inputs=[], outputs=build_output)

        question_input = gr.Textbox(label="Ask Question")
        answer_output = gr.Textbox(label="Answer")
        question_input.submit(fn=ask_graph_question, inputs=question_input, outputs=answer_output)

    with gr.Tab("Graph Visualization"):
        vis_btn = gr.Button("Draw RDF Graph")
        vis_output = gr.Image(type="filepath")
        vis_btn.click(fn=draw_graph, inputs=[], outputs=vis_output)

demo.launch()
